# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will use a simple action score to prioritize pages that have meaningful search exposure but show weaker recent performance or weaker search position. The baseline is intended as a transparent review queue, not as a prediction of future rankings.

The reason codes will identify why a page received a high score: HIGH_IMPRESSIONS, WEAK_POSITION, and RECENT_DECLINE. A page can receive more than one reason code when multiple signals support review.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Baseline rule:")
print("- Higher impressions increase review priority.")
print("- Weaker average position increases review priority.")
print("- Recent decline increases review priority.")

print("\nReason codes:")
print("HIGH_IMPRESSIONS")
print("WEAK_POSITION")
print("RECENT_DECLINE")

Baseline rule:
- Higher impressions increase review priority.
- Weaker average position increases review priority.
- Recent decline increases review priority.

Reason codes:
HIGH_IMPRESSIONS
WEAK_POSITION
RECENT_DECLINE


## 2. Build the ranked queue (writes the CSV)

The baseline score combines three observable signals: search exposure, average position, and recent change in impressions. The score is only used to rank pages for review. It does not represent a probability or causal estimate.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Recent impression change
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"],
    0
)

# Convert signals to percentile ranks.
# Higher value = higher review priority.
df["exposure_score"] = df["impressions_90d"].rank(pct=True)

# A larger average position means a weaker search position.
df["position_score"] = df["avg_position"].rank(pct=True)

# More negative recent change = more concerning.
df["decline_score"] = (-df["impression_change_pct"]).rank(pct=True)

# Combined transparent baseline score.
df["baseline_score"] = (
    0.40 * df["exposure_score"]
    + 0.35 * df["position_score"]
    + 0.25 * df["decline_score"]
)

# Reason codes
def reason_codes(row):
    reasons = []

    if row["exposure_score"] >= 0.75:
        reasons.append("HIGH_IMPRESSIONS")

    if row["position_score"] >= 0.75:
        reasons.append("WEAK_POSITION")

    if row["impression_change_pct"] < 0:
        reasons.append("RECENT_DECLINE")

    if not reasons:
        reasons.append("MIXED_SIGNAL")

    return "|".join(reasons)

df["reason_code"] = df.apply(reason_codes, axis=1)

# Rank highest-priority pages first
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Output columns
output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "impressions_last_30d",
    "impressions_prev_30d",
    "impression_change_pct"
]

baseline_output = df[output_columns].copy()

# Save output
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_output.to_csv(output_path, index=False)

print("Rows ranked:", len(baseline_output))
print("Output written to:", output_path)

display(baseline_output.head(20))

Rows ranked: 30000
Output written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,reason_code,impressions_90d,avg_position,impressions_last_30d,impressions_prev_30d,impression_change_pct
0,1,content_fb66dd8f4629,0.955048,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,32518,56.5,3725,25453,-0.853652
1,2,content_fb4bf6555c79,0.951901,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,84093,45.6,4300,25249,-0.829696
2,3,content_150f89b1d73b,0.949842,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,83490,45.0,5797,32027,-0.818996
3,4,content_b80d73524f2d,0.939144,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,35620,44.0,2380,12366,-0.807537
4,5,content_a7c2dfc8a6ec,0.935393,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,76868,39.9,7653,33908,-0.774301
5,6,content_b51e2e4d22ff,0.934647,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,91795,41.2,11283,44399,-0.745873
6,7,content_095661034f9b,0.934092,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,23513,39.4,952,11326,-0.915946
7,8,content_21b3d827d451,0.931249,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,38894,39.4,3427,16654,-0.794224
8,9,content_4a50087c06cb,0.930161,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,40971,47.1,4280,14283,-0.700343
9,10,content_370de6e8e035,0.929928,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,114389,39.7,12504,45323,-0.724114


## 3. Top-20 review

I will review the top 20 ranked pages as decision-support rather than treating the ranking as ground truth. Each page receives an action, a reason code, a confidence note, and a condition that could make the recommendation wrong. The recommended action is to review the page rather than automatically change it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline_output.head(20).copy()

top20["action"] = "REVIEW_CONTENT"

top20["confidence_note"] = np.where(
    top20["reason_code"].str.contains(r"\|"),
    "Multiple signals support review.",
    "Single primary signal supports review."
)

top20["what_could_make_it_wrong"] = (
    "Observed signals may not reflect the true content opportunity; "
    "manual review is required."
)

top20 = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_could_make_it_wrong"
    ]
]

display(top20)

,rank,content_id,action,reason_code,baseline_score,confidence_note,what_could_make_it_wrong
0,1,content_fb66dd8f4629,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.955048,Multiple signals support review.,Observed signals may not reflect the true cont...
1,2,content_fb4bf6555c79,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.951901,Multiple signals support review.,Observed signals may not reflect the true cont...
2,3,content_150f89b1d73b,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.949842,Multiple signals support review.,Observed signals may not reflect the true cont...
3,4,content_b80d73524f2d,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.939144,Multiple signals support review.,Observed signals may not reflect the true cont...
4,5,content_a7c2dfc8a6ec,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.935393,Multiple signals support review.,Observed signals may not reflect the true cont...
5,6,content_b51e2e4d22ff,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.934647,Multiple signals support review.,Observed signals may not reflect the true cont...
6,7,content_095661034f9b,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.934092,Multiple signals support review.,Observed signals may not reflect the true cont...
7,8,content_21b3d827d451,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.931249,Multiple signals support review.,Observed signals may not reflect the true cont...
8,9,content_4a50087c06cb,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.930161,Multiple signals support review.,Observed signals may not reflect the true cont...
9,10,content_370de6e8e035,REVIEW_CONTENT,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,0.929928,Multiple signals support review.,Observed signals may not reflect the true cont...


## 4. Weak picks + leakage check

Some high-ranked pages may be weak recommendations because high impressions or weak average position do not necessarily mean that content needs improvement. A page may intentionally target a difficult query, have a strong business reason for its current position, or have insufficient evidence of a genuine content problem. Therefore the baseline should only create a review queue.

I also checked that the baseline does not use content_id, client_id, product flags, or an explicitly future performance window. The recent-change signal uses the available last-30-day and previous-30-day measurements and should be treated as observed historical information.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Leakage check ===")

for field in ["content_id", "client_id"]:
    print(
        f"{field} used in score:",
        field in ["impressions_90d", "avg_position",
                  "impressions_last_30d", "impressions_prev_30d"]
    )

print("\nScore inputs:")
print([
    "impressions_90d",
    "avg_position",
    "impressions_last_30d",
    "impressions_prev_30d"
])

print("\nFuture-looking fields used: None explicitly.")
print("Product/client identifiers used: None.")

print("\nPotential weak-pick review:")
display(
    baseline_output.head(5)[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "avg_position",
            "impression_change_pct"
        ]
    ]
)

=== Leakage check ===
content_id used in score: False
client_id used in score: False

Score inputs:
['impressions_90d', 'avg_position', 'impressions_last_30d', 'impressions_prev_30d']

Future-looking fields used: None explicitly.
Product/client identifiers used: None.

Potential weak-pick review:


,rank,content_id,baseline_score,reason_code,avg_position,impression_change_pct
0,1,content_fb66dd8f4629,0.955048,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,56.5,-0.853652
1,2,content_fb4bf6555c79,0.951901,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,45.6,-0.829696
2,3,content_150f89b1d73b,0.949842,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,45.0,-0.818996
3,4,content_b80d73524f2d,0.939144,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,44.0,-0.807537
4,5,content_a7c2dfc8a6ec,0.935393,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,39.9,-0.774301


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.